# Exact-tier Global LoLalytics benchmarks — broad NA cohort, patch 16.19

Pinned source: **20261027T153245042167Z**, 300 completed players (50 each Bronze–Diamond), 5,706 target player-games. This notebook plans and joins only target-game played/eligible-alternative lookups. Master analysis, Riot checkpoints, and source Parquets are read-only.

**Collection is disabled by default.** The implementation was validated with cached fixtures and mocked responses only; exact-tier live access and continued historical availability have not been tested. Existing project receipts from 2026-10-20 confirm ordinary champion pages exposed complete named Qwik opponent tables. No public API is assumed, no directional fallback is used, and no access controls are bypassed.

## 1. Imports and configuration

Requires the existing environment's pandas, pyarrow, requests, beautifulsoup4, and IPython. No packages are installed automatically. Paths work from the repository root or `hlr_model/`.

The only collection switch is `RUN_COLLECTION`. Set it to True **and save the notebook** before Run All. Keep the source snapshot, fingerprint, and cache folder unchanged when resuming. `sparse_below_games=100` flags samples for reporting only: **there is no minimum-game exclusion**.

In [1]:
from pathlib import Path
import sys, json, importlib
import pandas as pd
from IPython.display import display

start = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [start, *start.parents]
    if (p / 'hlr_model/broad_rank_pipeline.py').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open from the repository root or hlr_model/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from hlr_model import broad_rank_matchups as matchups
importlib.reload(matchups)

RUN_COLLECTION = True
CONFIG = matchups.Config(
    source_snapshot='20261027T153245042167Z',
    source_export_fingerprint='352a75c162ac4f3ac573385b4c1b5cc9fe90e23b6c7d3af7b6e78ca1da4c9b44',
    run_name='broad_na_16_19_seed_161926_exact_tiers_v1',
    sparse_below_games=100,
    delay_seconds=3.0, max_attempts=3,
)
print('Collection enabled:', RUN_COLLECTION)
print('Filters: patch 16.19 | GLOBAL | Ranked Solo/Duo (420) | individual exact tiers')
print('Exact tiers:', ', '.join(CONFIG.tiers))

Collection enabled: True
Filters: patch 16.19 | GLOBAL | Ranked Solo/Duo (420) | individual exact tiers
Exact tiers: BRONZE, SILVER, GOLD, PLATINUM, EMERALD, DIAMOND


## 2. Load the explicitly selected completed source

Read and verify all five Parquet hashes against the completion metadata, cohort quotas, target/model identity, completed history flags, and time ordering. The source snapshot and all file hashes are frozen into this collector's identity; it never selects a newer snapshot automatically.

In [2]:
bundle = matchups.load_snapshot(PROJECT_ROOT, CONFIG)
print('Selected snapshot:', bundle['path'].relative_to(PROJECT_ROOT))
print('Export fingerprint:', bundle['identity']['export_fingerprint'])
display(bundle['frames']['player_cohort'].groupby('seed_tier').size().reindex(CONFIG.tiers).rename('players').to_frame())
display(pd.DataFrame([
    {'source_table': name, 'rows': len(frame), 'columns': len(frame.columns)}
    for name, frame in bundle['frames'].items()
]))

Selected snapshot: hlr_model/data/runs/broad_na_16_19_seed_161926/processed/20261027T153245042167Z
Export fingerprint: 352a75c162ac4f3ac573385b4c1b5cc9fe90e23b6c7d3af7b6e78ca1da4c9b44


,players
seed_tier,
BRONZE,50
SILVER,50
GOLD,50
PLATINUM,50
EMERALD,50
DIAMOND,50


,source_table,rows,columns
0,player_cohort,300,28
1,historical_matches,32746,21
2,target_matches,5706,21
3,longitudinal_modeling,5706,73
4,player_coverage,300,15


## 3. Build and summarize required played/alternative lookups

Reuse the existing project's exact candidate algorithm: preserved initial history plus earlier target games whose **end_timestamp < target kickoff**, same focal role, at least one prior play, excluding the currently played champion and the known target opponent. There is no proficiency cutoff and no requirement to improve the matchup rate. The played champion is requested separately even with zero prior experience.

Unknown role/opponent and unexpected mirrors remain in the target dataset and coverage denominator. Only played champions have observed outcomes; alternatives have missing candidate outcomes. Previously played does not imply draft availability.

Deduplicate directional lookups by patch, exact tier, Global, queue, champion, focal role, opponent and opposing role. Each champion/role/tier page is requested once for its complete opponent table, never once per player-game.

In [3]:
plan = matchups.build_plan(bundle, CONFIG)
print(f"Target player-games: {len(bundle['frames']['target_matches']):,}")
print(f"Unique required tables: {len(plan['tables']):,}")
print(f"Unique directional matchups: {len(plan['lookups']):,}")
display(plan['mapping'].request_kind.value_counts().rename('player_game_lookups').to_frame())
summary = plan['tables'].groupby(['tier', 'lane']).size().rename('unique_tables').reset_index()
display(summary.pivot(index='tier', columns='lane', values='unique_tables').reindex(CONFIG.tiers).fillna(0).astype(int))
print('Unreviewed champion mappings:', int((~plan['lookups'].mapping_verified_locally).sum()))

Target player-games: 5,706
Unique required tables: 1,798
Unique directional matchups: 28,936


,player_game_lookups
request_kind,
alternative,47356
observed,5706


lane,adc,jungle,mid,support,top
tier,,,,,
BRONZE,47,46,60,62,53
SILVER,41,56,55,52,79
GOLD,54,61,64,72,61
PLATINUM,41,62,75,72,79
EMERALD,50,66,72,57,60
DIAMOND,48,68,63,45,77


Unreviewed champion mappings: 0


## 4. Load checkpoint and show progress

The separate folder below contains SQLite configuration/table statuses, immutable per-attempt receipts, content-addressed HTML, and versioned exports. Selected filters, returned route/header, champion identity, both roles, schema, timestamps and body checksums are validated before cached data are reused. Failed requests remain unresolved.

Compatible existing receipt caches are read-only inputs. Exact URL **and** independently verified returned data must match; Master+ receipts cannot become exact-tier benchmarks. Interrupted temporary writes are ignored; complete receipts can recover an interrupted index/database update.

In [4]:
if 'collector' in globals():
    collector.close()
collector = matchups.Collector(PROJECT_ROOT, bundle, plan, CONFIG)
reused = collector.reuse_compatible([
    PROJECT_ROOT / 'hlr_model/data/lolalytics/runs',
    PROJECT_ROOT / 'lpm_model/data/lolalytics',
])
print('Separate cache/checkpoint:', collector.run_dir.relative_to(PROJECT_ROOT))
print('Compatible existing tables imported:', reused)
display(collector.show())

Separate cache/checkpoint: hlr_model/data/lolalytics/broad_na_16_19_seed_161926_exact_tiers_v1
Compatible existing tables imported: 0
Required tables=1,798; cached/validated=0; pending=1,798; failed=0; downloaded this session=0.
Table statuses: {'pending': 1798}; retry/cooldown: {}
Requested unique matchups: {'pending': 28936}


,tier,lane,status,tables
0,BRONZE,adc,pending,47
1,BRONZE,jungle,pending,46
2,BRONZE,mid,pending,60
3,BRONZE,support,pending,62
4,BRONZE,top,pending,53
5,DIAMOND,adc,pending,48
6,DIAMOND,jungle,pending,68
7,DIAMOND,mid,pending,63
8,DIAMOND,support,pending,45
9,DIAMOND,top,pending,77


## 5. Run/resume LoLalytics collection

Change the single switch to True, save, and Run All. Interrupt safely; after reopening or restarting the kernel, Run All resumes the same pinned plan and pending tables. Set False for offline previews/joins. No Riot key or Riot API requests are involved.

Ordinary sequential champion pages are used with 3-second pacing, bounded transient retries with capped backoff/jitter, and persisted Retry-After cooldowns. Redirects, challenges, access denials, wrong filters, unknown schemas, and unavailable historical patches stop clearly. URL parameters alone never establish filters. The known-broken `queue=ranked` URL parameter is omitted; selected Solo/Duo controls plus returned dataset queue 420 are required.

**If automation is blocked but the page is ordinarily accessible:** use `plan['tables']` to open its exact `source_url` in your normal browser, check selected filters, and Save Page Source as UTF-8 HTML. Import the saved source with the call below (replace the example path/time). The parser requires the complete named Qwik opponent table and matching selected filters, route and header. A screenshot or manually declared filters alone cannot establish valid source data. If patch 16.19 is no longer available through normal access, the work remains unresolved; this notebook never substitutes another patch, rank, region, or 50%.

```python
table = plan['tables'].query("tier == 'BRONZE' and lane == 'mid'").iloc[0]
print(table.source_url)
collector.import_page(
    table.table_id, Path('/absolute/path/to/saved-page.html'),
    source_url=table.source_url,
    retrieved_at='2026-10-27T12:00:00-07:00',  # actual browser retrieval time
)
```

Rerun the progress/export cells after imports. Preserve the failed receipts as evidence; successful imports create new receipts and do not edit the source caches.

In [5]:
collector.collect(enabled=RUN_COLLECTION)
display(collector.show())

Retry/pacing cooldown: 2.8s; interrupt safely to pause.
Retry/pacing cooldown: 2.8s; interrupt safely to pause.
Retry/pacing cooldown: 2.8s; interrupt safely to pause.
Retry/pacing cooldown: 2.8s; interrupt safely to pause.
Retry/pacing cooldown: 2.8s; interrupt safely to pause.
Retry/pacing cooldown: 2.8s; interrupt safely to pause.
Retry/pacing cooldown: 2.8s; interrupt safely to pause.
Retry/pacing cooldown: 2.8s; interrupt safely to pause.
Retry/pacing cooldown: 2.8s; interrupt safely to pause.
Retry/pacing cooldown: 2.8s; interrupt safely to pause.
Retry/pacing cooldown: 2.8s; interrupt safely to pause.
Retry/pacing cooldown: 2.8s; interrupt safely to pause.
Retry/pacing cooldown: 2.8s; interrupt safely to pause.
Retry/pacing cooldown: 2.8s; interrupt safely to pause.
Retry/pacing cooldown: 2.8s; interrupt safely to pause.
Retry/pacing cooldown: 2.8s; interrupt safely to pause.
Retry/pacing cooldown: 2.8s; interrupt safely to pause.
Retry/pacing cooldown: 2.8s; interrupt safely to

,tier,lane,status,tables
0,BRONZE,adc,validated,47
1,BRONZE,jungle,validated,46
2,BRONZE,mid,validated,60
3,BRONZE,support,validated,62
4,BRONZE,top,validated,53
5,DIAMOND,adc,validated,48
6,DIAMOND,jungle,validated,68
7,DIAMOND,mid,validated,63
8,DIAMOND,support,validated,45
9,DIAMOND,top,validated,77


## 6. Export lookup tables and joined datasets

Every export creates a new timestamped folder under this collector's `processed/`. Outputs include benchmark lookups, all played target-games, eligible alternatives, coverage/missingness, table statuses, required lookups, mapping, configuration, and pinned source metadata. Many-to-one/one-to-one join checks prevent row multiplication; original `puuid`/`match_id` and source row identifiers are preserved.

Exports are explicitly **PARTIAL** while any request/table/mapping remains unresolved. A validated complete table lacking an opponent is separately `not_published_in_table`, with unknown sample/rate. A published zero-game row and a sparse positive-count estimate have separate flags. No player-game is dropped because its benchmark is missing.

In [6]:
output_path, coverage, export_metadata = collector.export()
print('Versioned output:', output_path.relative_to(PROJECT_ROOT))
print('Collection complete:', export_metadata['collection_complete'])
display(pd.Series(export_metadata['row_counts'], name='rows').to_frame())
display(pd.Series(export_metadata['lookup_status_counts'], name='unique_matchups').to_frame())

COMPLETE: versioned export /Users/minhho-hoang/Documents/riot_proj/riot-game-prediction/hlr_model/data/lolalytics/broad_na_16_19_seed_161926_exact_tiers_v1/processed/20261027T194148939685Z_3b8d192e
Versioned output: hlr_model/data/lolalytics/broad_na_16_19_seed_161926_exact_tiers_v1/processed/20261027T194148939685Z_3b8d192e
Collection complete: True


,rows
played_before,5706
played_after,5706
alternatives_before,47356
alternatives_after,47356


,unique_matchups
ok,28876
not_published_in_table,60


## 7. Coverage, units, and study limits

Coverage uses every source target player-game and every eligible alternative, including missing/unverified benchmarks. Review exact-tier/role/status counts below.

Raw `*_pct` rates are percentages (0–100); `raw_matchup_winrate_probability` is raw percent / 100 only for a verified positive-count row. Counts are retained; wins/losses are missing because the reviewed source table does not expose them directly. They are never reconstructed from rounded percentages. `*_pp` source deltas are percentage points. Full original opponent-row values/names are retained in `matchup_evidence_json`.

The existing project adjustment is preserved separately: **raw_matchup_winrate_pct − tier_average_winrate_pct + 50**. It is a derived percentage-unit value, is not clipped, and is not a probability. It remains missing when the rendered exact-bracket average is absent. Source header `avgWr` is retained under its own name and is **not** substituted for the bracket average. Published Delta 2 keeps the source definition: counter matchup win rate minus normalised champions win rate in matchup. No reverse-direction complement is assumed.

Player rank labels were checked at collection time, not historically verified for patch 16.19. Full-patch matchup benchmarks are retrospective aggregates and were not necessarily available before each target game; this is not a claim of leakage-free prospective evaluation. Historical candidate play does not prove draft availability or the effect of switching champions. No models, mastery groups, or synergy analysis are included.

In [7]:
display(coverage.sort_values(['dataset', 'tier', 'lane', 'status']).reset_index(drop=True))
print('Selected source remains:', CONFIG.source_snapshot)
print('Complete only when every required table/mapping is verified; unavailable matchups are reported separately.')
for limitation in matchups.LIMITS:
    print('•', limitation)

,tier,lane,status,rows,valid_rows,sparse_rows,dataset,denominator_rows,fraction
0,BRONZE,adc,not_published_in_table,4,0,0,alternative,769,0.005202
1,BRONZE,adc,ok,765,765,111,alternative,769,0.994798
2,BRONZE,jungle,not_published_in_table,2,0,0,alternative,721,0.002774
3,BRONZE,jungle,ok,719,719,89,alternative,721,0.997226
4,BRONZE,mid,not_published_in_table,2,0,0,alternative,1024,0.001953
...,...,...,...,...,...,...,...,...,...
80,SILVER,adc,ok,149,149,23,played,149,1.000000
81,SILVER,jungle,ok,298,298,11,played,298,1.000000
82,SILVER,mid,ok,88,88,5,played,88,1.000000
83,SILVER,support,ok,125,125,1,played,125,1.000000


Selected source remains: 20261027T153245042167Z
Complete only when every required table/mapping is verified; unavailable matchups are reported separately.
• Player tier was checked at collection time, not verified historically during patch 16.19.
• Full-patch benchmarks are retrospective aggregates, not necessarily available before each target game.
• Prior same-role play establishes candidate eligibility, not draft availability or a causal switching effect.
• Roles follow postgame Riot teamPosition inference; no rank, region, patch or 50% fallback is used.
